In [5]:
from triples import triples, predicate_definitions_mapping
import re
import pandas as pd

df = pd.DataFrame.from_dict(predicate_definitions_mapping, orient="index",)
df.index.name = "canonical_predicate"

def extract_construct(text: str) -> str | None:
    if not isinstance(text, str):
        return None

    match = re.search(r":([^\s(]+)", text)
    return match.group(1) if match else None

for col in ["uml_equivalent", "sysml_equivalent"]:
    df[f"{col}_construct"] = df[col].apply(extract_construct)

df = df.reset_index()
df.head()

,canonical_predicate,definition,uml_equivalent,sysml_equivalent,uml_equivalent_construct,sysml_equivalent_construct
0,assembledAt,Vehicle model is assembled at a manufacturing ...,uml:Association (Directed Dependency or Genera...,sysml:Allocate (or Directed Allocation Relatio...,Association,Allocate
1,hasEngine,Vehicle model is equipped with a specific engi...,uml:Property (Composition / Composite Aggregat...,sysml:PartProperty (Block Definition Diagram c...,Property,PartProperty
2,hasFuelType,Vehicle model utilizes a primary fuel or propu...,uml:Property (Value Attribute / Enumeration Re...,sysml:ValueProperty (Typed by a ValueType),Property,ValueProperty
3,headquarteredIn,Organization or corporation is headquartered i...,uml:Association (Directed Association / Depend...,sysml:DirectedRelationship (or Item Flow / Dep...,Association,DirectedRelationship
4,manufactures,Automotive manufacturer produces a specific ve...,uml:Association (Navigable Directed Association),sysml:DirectedRelationship (or Action / Item F...,Association,DirectedRelationship


In [6]:
triples_df = pd.DataFrame(triples, columns=["subject", "predicate", "object"],)
mapped_triples = triples_df.merge(df, left_on="predicate", right_on="canonical_predicate", how="left", indicator=True,)
mapped_triples["mapping_status"] = mapped_triples["_merge"].map({"both": "mapped", "left_only": "unmapped",})
mapped_triples = mapped_triples.drop(columns=["_merge", "canonical_predicate"])
mapped_triples.head()

,subject,predicate,object,definition,uml_equivalent,sysml_equivalent,uml_equivalent_construct,sysml_equivalent_construct,mapping_status
0,Ford,manufactures,Mustang,Automotive manufacturer produces a specific ve...,uml:Association (Navigable Directed Association),sysml:DirectedRelationship (or Action / Item F...,Association,DirectedRelationship,mapped
1,Ford,manufactures,F-150,Automotive manufacturer produces a specific ve...,uml:Association (Navigable Directed Association),sysml:DirectedRelationship (or Action / Item F...,Association,DirectedRelationship,mapped
2,Toyota,manufactures,Camry,Automotive manufacturer produces a specific ve...,uml:Association (Navigable Directed Association),sysml:DirectedRelationship (or Action / Item F...,Association,DirectedRelationship,mapped
3,Toyota,manufactures,Corolla,Automotive manufacturer produces a specific ve...,uml:Association (Navigable Directed Association),sysml:DirectedRelationship (or Action / Item F...,Association,DirectedRelationship,mapped
4,Chevrolet,manufactures,Silverado,Automotive manufacturer produces a specific ve...,uml:Association (Navigable Directed Association),sysml:DirectedRelationship (or Action / Item F...,Association,DirectedRelationship,mapped


In [8]:
import uuid
import xml.etree.ElementTree as ET


XMI_NS = "http://www.omg.org/spec/XMI/20131001"
UML_NS = "http://www.omg.org/spec/UML/20110701"

ET.register_namespace("xmi", XMI_NS)
ET.register_namespace("uml", UML_NS)


def new_id():
    return f"_{uuid.uuid4().hex}"


def export_xmi(mapped_triples, output_path):
    xmi = f"{{{XMI_NS}}}"
    uml = f"{{{UML_NS}}}"

    root = ET.Element(f"{xmi}XMI", {f"{xmi}version": "2.5"},)
    model = ET.SubElement(root, f"{uml}Model", {f"{xmi}id": new_id(), "name": "ReconstructedModel",},)

    # One UML Class per distinct subject/object.
    class_ids = {}

    def get_class_id(name):
        if name not in class_ids:
            element_id = new_id()
            class_ids[name] = element_id

            ET.SubElement(model, "packagedElement", 
                          {f"{xmi}type": "uml:Class", f"{xmi}id": element_id, "name": str(name),},
                          )

        return class_ids[name]

    for row in mapped_triples.itertuples(index=False):
        if row.mapping_status != "mapped":
            continue

        if row.uml_equivalent_construct != "Association":
            continue

        subject_id = get_class_id(row.subject)
        object_id = get_class_id(row.object)

        association_id = new_id()
        source_end_id = new_id()
        target_end_id = new_id()

        association = ET.SubElement(
            model,
            "packagedElement",
            {
                f"{xmi}type": "uml:Association", 
                f"{xmi}id": association_id, 
                "name": str(row.predicate), 
                "memberEnd": f"{source_end_id} {target_end_id}",
                },
        )

        ET.SubElement(
            association,
            "ownedEnd",
            {
                f"{xmi}type": "uml:Property",
                f"{xmi}id": source_end_id,
                "name": "source",
                "type": subject_id,
                "association": association_id,
            },
        )

        ET.SubElement(
            association,
            "ownedEnd",
            {
                f"{xmi}type": "uml:Property",
                f"{xmi}id": target_end_id,
                "name": "target",
                "type": object_id,
                "association": association_id,
            },
        )

    ET.ElementTree(root).write(
        output_path,
        encoding="UTF-8",
        xml_declaration=True,
    )

In [ ]:
from uuid import uuid4

def create_mapping_table(conn):
    with conn.cursor() as cur:
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS predicate_mappings (
                language TEXT NOT NULL
                    CHECK (language IN ('UML', 'SYSML')),
                construct TEXT NOT NULL,
                definition TEXT,
                mapping_details TEXT,
                goverance_status TEXT NOT NULL
                    CHECK (governance_status IN ('DRAFT', 'ACCEPTED')),
                predicate_uuid UUID NOT NULL
                    REFERENCES predicates(predicate_uuid),
                mapping_uuid UUID PRIMARY KEY
            )
            """
        )


def load_predicate_mappings(conn, mapping_definitions):
    with conn.cursor() as cur:
        for predicate, details in mapping_definitions.items():
            cur.execute(
                """
                SELECT predicate_uuid
                FROM predicates
                WHERE predicate = %s
                """,
                (predicate,),
            )
            result = cur.fetchone()

            if result is None:
                raise ValueError(f"Predicate {predicate!r} does not exist in the predicates table.")

            predicate_uuid = result[0]

            for language in ("uml", "sysml"):
                mapping = details.get(f"{language}_equivalent")

                if not mapping:
                    continue

                # Extract the namespace-qualified construct.
                construct = mapping.split(":", 1)[1].split()[0]

                cur.execute(
                    """
                    INSERT INTO predicate_mappings (
                        language,
                        construct,
                        definition,
                        mapping_details,
                        predicate_uuid,
                        mapping_uuid
                    )
                    VALUES (%s, %s, %s, %s, %s, %s)
                    ON CONFLICT (predicate_uuid, language, construct)
                    DO NOTHING
                    """,
                    (
                        language.upper(),
                        construct,
                        details.get("definition"),
                        mapping,
                        predicate_uuid,
                        str(uuid4())
                    ),
                )